# **README**

# **GUIDE:**
https://drive.google.com/file/d/1TdwyZxW1EMlDhzg3akTqwosRQDM7D62Z/view?usp=sharing

# Arabic Audio Transcriber

Transcribes Arabic (and mixed Arabic/English) audio using **Faster-Whisper** on a free Colab GPU.
THIS VERSION IS FOR DIRECTLY SUBTITLING VIDEOS, FOR RAW TRANSLATION USE THE OTHER VERSION

**Supports:**
- Upload a local audio/video file (MP3, MP4, WAV, etc.)
- Paste a YouTube URL to download and transcribe (audio-only or full MP4)
- Generate an **SRT subtitle file** from the transcript
- **Burn subtitles into the video** with ffmpeg

**Outputs:**
- `<name>_transcript.txt` — plain text transcript
- `<name>_transcript_timestamps.txt` — transcript with `[MM:SS-MM:SS]` markers
- `<name>.srt` *(optional)* — subtitle file
- `<name>_captioned.mp4` *(optional)* — original video with burned-in captions

---
**Before running:** go to `Runtime → Change runtime type` and select **T4 GPU** for best speed.

# Steps:
1. Run cells 1-3
2. Select 4a if uploading mp3 (you can use any online tool to convert mp4 to mp3)
3. Select 4b if using a youtube video (replace the link with your desired video)
4. Run 5-6

# For Translating (As written response, see below for generating captions)

You can upload the transcript to the following chatgpt https://chatgpt.com/g/g-FquaMCvw7-salafi-research-tool

# For Generating Captions Directly on Video

## Captions under 10 minutes:

"
Translate the attached transcript into English. Preserve the intended meaning and word choice as closely as possible.
This is a generated transcript, so it may contain minor errors. Use the surrounding context to correct obvious transcription errors, but do not make large interpretive changes. For context, this is Islamic material.

Format the output as a timestamped transcript, for example:
[00:00-00:05] Hello everyone.

Timestamp rules:
Keep the original timestamp blocks and their order. Do not invent new timestamps, remove timestamps, extend timestamps, shorten timestamps, or change the start/end time of any timestamp block.
The translated transcript must begin and end at exactly the same timestamps as the source chunk.
You may only adjust caption balance by moving translated words between adjacent existing timestamp blocks when the speech clearly continues across them. Do not create new time ranges.
If a source line is very short, keep it short unless moving nearby words into it is necessary for readability.
If a source line is very long, keep the same timestamp and translate it fully; do not split it into new timestamps.

Output rules:
Do not include notes, explanations, headings, corrections, or commentary.
Do not include anything other than the translated transcript.
Preserve the original wording as much as possible.

REMINDER: PRESERVE ORIGINAL WORDING AND ORIGINAL TIMESTAMP BOUNDARIES.
"

## Captions over 10 minutes:

"
Translate the attached transcript into English. Preserve the intended meaning and word choice as closely as possible.
This is a generated transcript, so it may contain minor errors. Use the surrounding context to correct obvious transcription errors, but do not make large interpretive changes. For context, this is Islamic material.

Format the output as a timestamped transcript, for example:
[00:00-00:05] Hello everyone.

Timestamp rules:
Keep the original timestamp blocks and their order. Do not invent new timestamps, remove timestamps, extend timestamps, shorten timestamps, or change the start/end time of any timestamp block.
The translated transcript must begin and end at exactly the same timestamps as the source chunk.
You may only adjust caption balance by moving translated words between adjacent existing timestamp blocks when the speech clearly continues across them. Do not create new time ranges.
If a source line is very short, keep it short unless moving nearby words into it is necessary for readability.
If a source line is very long, keep the same timestamp and translate it fully; do not split it into new timestamps.

Continuity:
This video is long, so I will provide it in 20-minute chunks. Treat each part as continuing from the previous part, not as an independent transcript. Preserve terminology consistently across all parts.


Output rules:
Do not include notes, explanations, headings, corrections, or commentary.
Do not include anything other than the translated transcript.
Preserve the original wording as much as possible.

REMINDER: PRESERVE ORIGINAL WORDING AND ORIGINAL TIMESTAMP BOUNDARIES.
"

In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────────
# hf_xet enables fast, chunked model downloads from Hugging Face (large-v3-turbo
# is served via Xet Storage — without hf_xet it falls back to slow HTTP).
!pip install -q -U faster-whisper yt-dlp yt-dlp-youtube-oauth2 hf_xet
# ffmpeg is required to merge video+audio streams (Cell 4B with DOWNLOAD_VIDEO=True)
!apt-get -y -qq install ffmpeg > /dev/null


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 64.2 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Check GPU ────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16"
else:
    print("⚠ No GPU found — running on CPU (slower). "
          "Go to Runtime → Change runtime type → T4 GPU for better speed.")
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8"

✓ GPU available: Tesla T4


## Configuration

Edit the values below before running the notebook.

| Setting | Options | Notes |
|---------|---------|-------|
| `MODEL_SIZE` | `tiny` `base` `small` `medium` `large-v2` `large-v3` `large-v3-turbo` | Larger = more accurate, slower. `large-v3-turbo` is nearly as accurate as `large-v3` but much faster |
| `LANGUAGE` | `"ar"`, `"en"`, `"fr"` … or `None` | `None` = auto-detect from first 30 s |
| `START_TIME` | `"8:00"`, `"1:25:00"`, `None` | Leave `None` for beginning of file |
| `END_TIME` | `"25:00"`, `None` | Leave `None` for end of file |

In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────

MODEL_SIZE  = "large-v3-turbo"   # tiny | base | small | medium | large-v2 | large-v3 | large-v3-turbo
LANGUAGE    = "ar"          # e.g. "ar", "en", "fr", or None to auto-detect

# Optional time range — set to None to transcribe the whole file
START_TIME  = None          # e.g. "8:00" or "1:25:00"
END_TIME    = None          # e.g. "25:00"

# ── Helpers ──────────────────────────────────────────────────────────────────
def _parse_ts(s):
    """Parse 'SS', 'M:SS', 'MM:SS', or 'H:MM:SS' into seconds (float)."""
    if s is None:
        return None
    parts = str(s).strip().split(":")
    nums  = [float(p) for p in parts]
    if len(nums) == 1:
        return nums[0]
    if len(nums) == 2:
        return nums[0] * 60 + nums[1]
    return nums[0] * 3600 + nums[1] * 60 + nums[2]

def _fmt_ts(t):
    """Format seconds as MM:SS or H:MM:SS."""
    s = int(round(t))
    h, r = divmod(s, 3600)
    m, sec = divmod(r, 60)
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m:02d}:{sec:02d}"

start_sec = _parse_ts(START_TIME)
end_sec   = _parse_ts(END_TIME)

print(f"Model   : {MODEL_SIZE}  |  Device: {DEVICE}  |  Compute: {COMPUTE_TYPE}")
print(f"Language: {LANGUAGE or 'auto-detect'}")
if start_sec is not None or end_sec is not None:
    s = _fmt_ts(start_sec or 0)
    e = _fmt_ts(end_sec)   if end_sec is not None else "end"
    print(f"Range   : {s} → {e}")
else:
    print("Range   : full file")

Model   : large-v2  |  Device: cuda  |  Compute: float16
Language: ar
Range   : full file


## Input — choose one of the two cells below

**Option A** — upload a file from your computer  
**Option B** — paste a YouTube URL

In [ ]:
# ── Cell 4A: Upload a local file ─────────────────────────────────────────────
# Run this cell OR Cell 4B — not both.

from google.colab import files as colab_files
from pathlib import Path

print("Select an audio or video file to upload (MP3, MP4, WAV, FLAC, M4A …)")
uploaded = colab_files.upload()

if not uploaded:
    raise SystemExit("No file uploaded — re-run this cell and choose a file.")

AUDIO_PATH = Path(next(iter(uploaded)))

# If the user uploaded a video, remember it so we can burn captions later
VIDEO_EXTS = {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v"}
VIDEO_PATH = AUDIO_PATH if AUDIO_PATH.suffix.lower() in VIDEO_EXTS else None

print(f"Uploaded: {AUDIO_PATH}  ({AUDIO_PATH.stat().st_size / 1e6:.1f} MB)")
if VIDEO_PATH is not None:
    print("Video file detected — captioning available after transcription.")

In [ ]:
# ── Cell 4B: Download from a YouTube URL ────────────────────────────────────────
# Run this cell OR Cell 4A — not both.
#
# AUTH OPTIONS (choose one):
#   1. USE_OAUTH = True  ← recommended: generates a device code; paste in browser once.
#   2. Upload cookies.txt and set COOKIES_FILE or UPLOAD_COOKIES_NOW = True.
#   3. Leave both off — works for public videos without bot checks.

import re
import subprocess
from pathlib import Path

YOUTUBE_URL    = "https://www.youtube.com/watch?v=XXXXXXXXXXX"  # ← paste URL
DOWNLOAD_VIDEO = True    # True = keep MP4; False = audio-only MP3

USE_OAUTH          = True   # OAuth2 device-code (most reliable on Colab)
COOKIES_FILE       = None   # e.g. "/content/yt_cookies.txt"
UPLOAD_COOKIES_NOW = False  # True → upload cookies.txt now


def _slugify(text: str, max_len: int = 60) -> str:
    text = text.strip().lower()
    text = re.sub(r"[^\w\s-]", "", text, flags=re.UNICODE)
    text = re.sub(r"[\s_-]+", "_", text)
    return text.strip("_")[:max_len]


_id_match = re.search(r"(?:v=|youtu\.be/|/shorts/)([A-Za-z0-9_-]{11})", YOUTUBE_URL)
VIDEO_ID = _id_match.group(1) if _id_match else None

BASE_OPTS = ["--no-playlist", "--no-warnings"]

# ── Auth args ──────────────────────────────────────────────────────────────
if UPLOAD_COOKIES_NOW:
    from google.colab import files as colab_files
    print("Upload your cookies.txt …")
    _up = colab_files.upload()
    if _up:
        _raw = next(iter(_up))
        _clean = Path("/content/yt_cookies.txt")
        Path(_raw).rename(_clean)
        COOKIES_FILE = str(_clean)
        print(f"Cookies saved as: {COOKIES_FILE}")

def _auth_args() -> list[str]:
    if USE_OAUTH:
        return ["--username", "oauth2", "--password", ""]
    if COOKIES_FILE and Path(COOKIES_FILE).exists():
        return ["--cookies", str(COOKIES_FILE)]
    return []

AUTH = _auth_args()
if USE_OAUTH:
    print("OAuth2 mode: yt-dlp will show a device code — visit the URL and approve it.")
elif AUTH:
    print(f"Cookie mode: {COOKIES_FILE}")
else:
    print("No auth — attempting anonymous download.")


# ── Helper ───────────────────────────────────────────────────────────────────
def _run_ytdlp(extra_args: list[str]) -> bool:
    cmd = ["yt-dlp", *BASE_OPTS, *AUTH, *extra_args, YOUTUBE_URL]
    result = subprocess.run(cmd, text=True, capture_output=False)
    if result.returncode != 0:
        return False
    return True


# ── Fetch title ─────────────────────────────────────────────────────────────
print("\nFetching video info …")
_meta = subprocess.run(
    ["yt-dlp", "--print", "%(title)s", "--no-download", *BASE_OPTS, *AUTH, YOUTUBE_URL],
    capture_output=True, text=True,
)
title = _meta.stdout.strip() if _meta.returncode == 0 else ""
STEM  = _slugify(title) if title else (VIDEO_ID or "youtube_download")
print(f"Title : {title or '(unknown)'}")
print(f"Stem  : {STEM}")
OUT_TEMPLATE = f"{STEM}.%(ext)s"

# ── Download ─────────────────────────────────────────────────────────────────
if DOWNLOAD_VIDEO:
    h   = MAX_VIDEO_HEIGHT if "MAX_VIDEO_HEIGHT" in dir() else 720
    cap = str(h)
    print(f"\nDownloading video (up to {h}p) …")
    video_formats = [
        ["-f", f"bestvideo[height<={cap}][ext=mp4]+bestaudio[ext=m4a]/bestvideo[height<={cap}]+bestaudio/best",
         "-S", f"res:{cap}", "--merge-output-format", "mp4"],
        ["-f", f"bestvideo[height<={cap}]+bestaudio/best", "-S", f"res:{cap}", "--merge-output-format", "mp4"],
        ["-f", "bv*+ba/b", "--merge-output-format", "mp4"],
        ["-f", "best[ext=mp4]/best", "--merge-output-format", "mp4"],
    ]
    ok = False
    for fmt_args in video_formats:
        print(f"  Format: {' '.join(fmt_args)}")
        if _run_ytdlp(["-o", OUT_TEMPLATE, *fmt_args]):
            ok = True
            break
    if not ok:
        raise SystemExit(
            "\nDownload failed.\n\n"
            "Tips:\n"
            "  • OAuth: did you approve the device code in your browser?\n"
            "  • Cookies: re-export from a browser actively logged into YouTube.\n"
            "  • Age-restricted/private videos: use Cell 4A to upload the file directly.\n"
            "  • Re-run Cell 1 to update: pip install -U yt-dlp yt-dlp-youtube-oauth2\n"
        )

    candidates = sorted(Path(".").glob(f"{STEM}.*"),
                        key=lambda p: (p.suffix.lower() != ".mp4", -p.stat().st_mtime))
    video_candidates = [p for p in candidates if p.suffix.lower() in {".mp4", ".mkv", ".webm", ".mov"}]
    if not video_candidates:
        raise SystemExit(f"No video file matching {STEM}.* found after download.")
    VIDEO_PATH = video_candidates[0]
    AUDIO_PATH = VIDEO_PATH
    print(f"Video saved: {VIDEO_PATH}  ({VIDEO_PATH.stat().st_size / 1e6:.1f} MB)")

else:
    print("\nDownloading audio …")
    if not _run_ytdlp(["-o", OUT_TEMPLATE, "-f", "bestaudio/best",
                        "--extract-audio", "--audio-format", "mp3", "--audio-quality", "0"]):
        raise SystemExit("Audio download failed. Check errors above.")

    mp3s = sorted(Path(".").glob(f"{STEM}.mp3")) or \
           sorted(Path(".").glob(f"{STEM}.*"), key=lambda p: -p.stat().st_mtime)
    if not mp3s:
        raise SystemExit(f"No audio file matching {STEM}.* found.")
    AUDIO_PATH = mp3s[0]
    VIDEO_PATH = None
    print(f"Audio saved: {AUDIO_PATH}  ({AUDIO_PATH.stat().st_size / 1e6:.1f} MB)")


Fetching video info …
Title: 1  شرح كتاب التوحيد   الشيخ   صالح آل الشيخ
Output files will be named: 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ.*

  Trying format: -f bv*+ba/b --merge-output-format mp4
Video saved: 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ.mp4  (72.2 MB)


In [ ]:
# ── Cell 5: Transcribe ───────────────────────────────────────────────────────
from faster_whisper import WhisperModel
from pathlib import Path

print(f"Loading Faster-Whisper '{MODEL_SIZE}' on {DEVICE} ({COMPUTE_TYPE}) …")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

# Build clip_timestamps argument
use_clip = start_sec is not None or end_sec is not None
clip_start = start_sec or 0.0

if not use_clip:
    clip_timestamps = "0"
    vad_filter = True
elif end_sec is None:
    clip_timestamps = str(clip_start)
    vad_filter = False
else:
    clip_timestamps = f"{clip_start},{float(end_sec)}"
    vad_filter = False

print(f"Transcribing {AUDIO_PATH.name} …")
segments, info = model.transcribe(
    str(AUDIO_PATH),
    language=LANGUAGE,
    beam_size=5,
    temperature=0.0,
    condition_on_previous_text=False,
    vad_filter=vad_filter,
    clip_timestamps=clip_timestamps,

    # Helpful for long-form transcription
    word_timestamps=False,
    compression_ratio_threshold=2.4,
    log_prob_threshold=-1.0,
    no_speech_threshold=0.6,
)

duration = getattr(info, "duration", None)
if duration:
    print(f"Audio length : {_fmt_ts(duration)}")
if use_clip:
    eff_end = min(end_sec, duration) if (end_sec and duration) else (end_sec or duration or 0)
    print(f"Transcribing : {_fmt_ts(clip_start)} → {_fmt_ts(eff_end)}")

# Collect segments and print as they stream in
texts       = []
timed_lines = []

for seg in segments:
    text = seg.text.strip()
    if not text:
        continue
    seg_start = float(seg.start or 0)
    seg_end   = float(seg.end   or 0)
    line = f"[{_fmt_ts(seg_start)}-{_fmt_ts(seg_end)}] {text}"
    texts.append(text)
    timed_lines.append(line)
    print(line)

full_text  = " ".join(texts)
timed_text = "\n".join(timed_lines)
print("\n✓ Transcription complete.")

Loading Faster-Whisper 'large-v2' on cuda (float16) …
Transcribing 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ.mp4 …
Audio length : 47:16
[00:00-00:02] بسم الله الرحمن الرحيم
[00:02-00:04] الحمد لله
[00:04-00:06] والصلاة والسلام على رسول الله
[00:06-00:09] يسر تسجيلات الراية الإسلامية
[00:09-00:11] أن تقدم لكم هذه المادة
[00:11-00:13] أولتها بعنوان
[00:13-00:16] شرح كتاب التوحيد للإمام المجدد
[00:16-00:18] محمد بن عبد الوهاب
[00:18-00:20] رحمه الله تعالى
[00:20-00:23] والذي قام بشرحه معاني الشيخ
[00:23-00:26] صالح بن عبد العزيز آل الشيخ
[00:26-00:28] في الدورة العلمية الثالثة
[00:28-00:33] بجامعِ شيخِ الإسلام إبريتيمية بمدينة الريام
[00:33-00:38] لعام ألفٍ وأربعمائة وستة عشر من الهجرةِ النبوية
[00:39-00:41] بسم الله الرحمن الرحيم
[00:42-00:43] الحمد لله
[00:44-00:48] الذي بعثَ عبادَهُ المرسلين بتوحيده
[00:49-00:51] فأقاموا الحُجَّةَ على العباد
[00:52-00:56] واتَّفَقوا من أولهم إلى آخرهم
[00:56-01:01] وأشهد أن لا إله إلا الله وحده لا شريك له
[01:01-01:07] تأكيدًا بعد تأكيد لبيان مقام التوحيد

In [ ]:
# ── Cell 6: Save and download results ────────────────────────────────────────
from google.colab import files as colab_files
from pathlib import Path

stem = AUDIO_PATH.stem[:50]

timed_path = Path(f"{stem}_transcript_timestamps.txt")

timed_path.write_text(timed_text, encoding="utf-8")

print(f"Timed transcript  → {timed_path}")
print()

# Preview first 2 000 characters
preview = timed_text[:2000]
print("──── Preview (first 2 000 chars) ────")
print(preview)
if len(timed_text) > 2000:
    print(f"… ({len(timed_text) - 2000} more characters)")

print("\nDownloading files …")
colab_files.download(str(timed_path))

Timed transcript  → 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ_transcript_timestamps.txt

──── Preview (first 2 000 chars) ────
[00:00-00:02] بسم الله الرحمن الرحيم
[00:02-00:04] الحمد لله
[00:04-00:06] والصلاة والسلام على رسول الله
[00:06-00:09] يسر تسجيلات الراية الإسلامية
[00:09-00:11] أن تقدم لكم هذه المادة
[00:11-00:13] أولتها بعنوان
[00:13-00:16] شرح كتاب التوحيد للإمام المجدد
[00:16-00:18] محمد بن عبد الوهاب
[00:18-00:20] رحمه الله تعالى
[00:20-00:23] والذي قام بشرحه معاني الشيخ
[00:23-00:26] صالح بن عبد العزيز آل الشيخ
[00:26-00:28] في الدورة العلمية الثالثة
[00:28-00:33] بجامعِ شيخِ الإسلام إبريتيمية بمدينة الريام
[00:33-00:38] لعام ألفٍ وأربعمائة وستة عشر من الهجرةِ النبوية
[00:39-00:41] بسم الله الرحمن الرحيم
[00:42-00:43] الحمد لله
[00:44-00:48] الذي بعثَ عبادَهُ المرسلين بتوحيده
[00:49-00:51] فأقاموا الحُجَّةَ على العباد
[00:52-00:56] واتَّفَقوا من أولهم إلى آخرهم
[00:56-01:01] وأشهد أن لا إله إلا الله وحده لا شريك له
[01:01-01:07] تأكيدًا بعد تأكيد لبيان مقام التوحيد
[01:07-0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Subtitles

Two follow-up steps you can run after a transcription is complete:

1. **Generate an SRT** from the timestamped transcript (either the one produced in this session, or a `.txt` file you upload).
*Feel free to edit/modify the .txt prior to uploading incase there are any translation errors*
2. **Burn the SRT into the video** with ffmpeg, producing a captioned MP4.

Captioning step 2 requires a video file (`VIDEO_PATH`). You get one by:
- Uploading an MP4 in **Cell 4A**, or
- Setting `DOWNLOAD_VIDEO = True` in **Cell 4B** before running it.

In [ ]:
# ── Cell 7: Generate an SRT from a timestamped transcript ────────────────────
#
# SRT_SOURCE controls where the input transcript comes from:
#   "session" — use the timestamps just produced by Cell 5
#   "upload"  — upload a .txt file with [MM:SS-MM:SS] or [H:MM:SS-H:MM:SS] lines

import re
from pathlib import Path

SRT_SOURCE = "upload"     # "session" or "upload"

if SRT_SOURCE == "upload":
    from google.colab import files as colab_files
    print("Upload a transcript file (lines like '[MM:SS-MM:SS] text') …")
    uploaded = colab_files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    src_path     = Path(next(iter(uploaded)))
    source_text  = src_path.read_text(encoding="utf-8")
    stem         = src_path.stem
else:
    if "timed_text" not in dir():
        raise SystemExit(
            "No transcript found in this session — run Cell 5 first, "
            "or set SRT_SOURCE = 'upload'."
        )
    source_text = timed_text
    stem        = AUDIO_PATH.stem[:50] if "AUDIO_PATH" in dir() else "captions"

# Matches both [MM:SS-MM:SS] and [H:MM:SS-H:MM:SS] (and even mixed in one file)
pattern = re.compile(r"\[(\d+(?::\d{2})+)\s*-\s*(\d+(?::\d{2})+)\]\s*(.*)")

def _ts_to_seconds(ts: str) -> float:
    nums = [int(p) for p in ts.split(":")]
    if len(nums) == 2:
        return nums[0] * 60 + nums[1]
    if len(nums) == 3:
        return nums[0] * 3600 + nums[1] * 60 + nums[2]
    raise ValueError(f"Cannot parse timestamp: {ts!r}")

def _seconds_to_srt(t: float) -> str:
    ms = int(round(t * 1000))
    h,  rem = divmod(ms, 3_600_000)
    m,  rem = divmod(rem,  60_000)
    s,  ms  = divmod(rem,   1_000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

entries = []
counter = 1
for line in source_text.splitlines():
    m = pattern.match(line.strip())
    if not m:
        continue
    start_raw, end_raw, text = m.groups()
    text = text.strip()
    if not text:
        continue
    start = _seconds_to_srt(_ts_to_seconds(start_raw))
    end   = _seconds_to_srt(_ts_to_seconds(end_raw))
    entries.append(f"{counter}\n{start} --> {end}\n{text}\n")
    counter += 1

if not entries:
    raise SystemExit(
        "No [MM:SS-MM:SS]/[H:MM:SS-H:MM:SS] lines found in the input. "
        "Check the file format."
    )

SRT_PATH = Path(f"{stem}.srt")
SRT_PATH.write_text("\n".join(entries), encoding="utf-8")

print(f"✓ Wrote {SRT_PATH}  ({counter - 1} captions)")
print()
print("──── Preview (first 1 000 chars) ────")
print(SRT_PATH.read_text(encoding="utf-8")[:1000])

from google.colab import files as colab_files

Upload a transcript file (lines like '[MM:SS-MM:SS] text') …


Saving 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ_transcript_timestamps.txt to 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ_transcript_timestamps (1).txt
✓ Wrote 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ_transcript_timestamps (1).srt  (574 captions)

──── Preview (first 1 000 chars) ────
1
00:00:00,000 --> 00:00:02,000
In the name of Allah, the Most Merciful, the Bestower of Mercy.

2
00:00:02,000 --> 00:00:04,000
All praise is for Allah.

3
00:00:04,000 --> 00:00:06,000
And prayers and peace be upon the Messenger of Allah.

4
00:00:06,000 --> 00:00:09,000
Al-Rayah Islamic Recordings is pleased

5
00:00:09,000 --> 00:00:11,000
to present to you this material,

6
00:00:11,000 --> 00:00:13,000
which it has titled:

7
00:00:13,000 --> 00:00:16,000
Explanation of Kitab al-Tawhid by the reviving imam,

8
00:00:16,000 --> 00:00:18,000
Muhammad ibn Abd al-Wahhab,

9
00:00:18,000 --> 00:00:20,000
may Allah the Exalted have mercy on him,

10
00:00:20,000 --> 00:00:23,000
whose meanings were explained by the

In [ ]:
# ── Cell 8: Burn the SRT into the video with ffmpeg ──────────────────────────
#
# Requires:
#   - SRT_PATH   (created by Cell 7)
#   - VIDEO_PATH (an MP4) — set automatically when you uploaded a video in
#                Cell 4A or used DOWNLOAD_VIDEO=True in Cell 4B.
#                If neither, you'll be prompted to upload one now.

import subprocess
from pathlib import Path
from google.colab import files as colab_files

if "SRT_PATH" not in dir():
    raise SystemExit("No SRT yet — run Cell 7 first.")

if "VIDEO_PATH" not in dir() or VIDEO_PATH is None:
    print("No video available in this session.")
    print("Upload an MP4 now, or cancel to skip captioning.")
    uploaded = colab_files.upload()
    if not uploaded:
        raise SystemExit("No video uploaded — skipping captioning.")
    VIDEO_PATH = Path(next(iter(uploaded)))

# Video encode quality (burning subs forces a re-encode; lower CRF = higher quality)
VIDEO_CRF    = 16       # 18–20 = high quality; default ffmpeg ~23 looks softer
VIDEO_PRESET = "slow" # ultrafast … veryslow (slower = better compression)

# Optional libass styling for the burned-in captions
# https://ffmpeg.org/ffmpeg-filters.html#subtitles-1
SUBTITLE_STYLE = (
    "FontName=Arial,"
    "FontSize=20,"
    "PrimaryColour=&H00FFFFFF,"    # white text (AABBGGRR)
    "BackColour=&H00000000,"       # ~50% transparent black box
    "BorderStyle=3,"               # 3 = opaque background box
    "MarginV=40,"
    "Alignment=2"                  # bottom center
)

OUT_VIDEO = Path(f"{VIDEO_PATH.stem}_captioned.mp4")

# ffmpeg's subtitles filter parses `:` and `,` specially — quote the style
filter_arg = f"subtitles={SRT_PATH.name}:force_style='{SUBTITLE_STYLE}'"

print(f"Burning {SRT_PATH.name} into {VIDEO_PATH.name} → {OUT_VIDEO.name}")
print(f"Encode: libx264 crf={VIDEO_CRF} preset={VIDEO_PRESET} (this may take a few minutes)")

result = subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(VIDEO_PATH),
        "-vf", filter_arg,
        "-c:v", "libx264",
        "-crf", str(VIDEO_CRF),
        "-preset", VIDEO_PRESET,
        "-c:a", "copy",
        str(OUT_VIDEO),
    ],
    capture_output=True, text=True,
)

if result.returncode != 0:
    print("ffmpeg failed:")
    print(result.stderr[-2000:])
    raise SystemExit(result.returncode)

print(f"✓ Wrote {OUT_VIDEO}  ({OUT_VIDEO.stat().st_size / 1e6:.1f} MB)")
colab_files.download(str(OUT_VIDEO))

Burning 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ_transcript_timestamps (1).srt into 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ.mp4 → 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ_captioned.mp4
Encode: libx264 crf=16 preset=slow (this may take a few minutes)
✓ Wrote 1_شرح_كتاب_التوحيد_الشيخ_صالح_آل_الشيخ_captioned.mp4  (74.9 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>